# Single Entrypoint: The `Codeflare` Class

This notebook demonstrates the new single entrypoint for the CodeFlare SDK.
Instead of managing authentication and configuration separately, everything
goes through one object:

```python
cf = Codeflare(config=SDKConfig(auth=..., namespace=...))
```

From there you use **handlers** to work with clusters and jobs:
- `cf.clusters.create(...)`, `cf.clusters.get(...)`, `cf.clusters.list()`
- `cf.jobs.submit(...)`, `cf.jobs.create(...)`

We will:
1. Authenticate and create the `Codeflare` entrypoint
2. Create and bring up a Ray cluster via `cf.clusters`
3. Submit a RayJob CR via `cf.jobs`
4. Clean up

## Step 1: Authenticate and create the entrypoint

Import `Codeflare` and `SDKConfig` from the SDK, and `AuthConfig` from kube-authkit.
Choose the authentication method that matches your environment.

In [ ]:
from codeflare_sdk import Codeflare, SDKConfig
from kube_authkit import AuthConfig

In [ ]:
# --- Pick ONE auth method and uncomment it ---

# Option 1: Auto-detect (kubeconfig or in-cluster service account)
# Works locally if you have ~/.kube/config, or inside a pod with a service account.
# auth_config = AuthConfig(method="auto")

# Option 2: Token-based (recommended for RHOAI Workbenches)
# Get your token: oc whoami -t
auth_config = AuthConfig(
    method="openshift",
    k8s_api_host="https://api.example.com:6443",
    token="sha256~XXXXX",  # oc whoami -t
)

# Option 3: OIDC (for BYOIDC-enabled clusters)
# auth_config = AuthConfig(
#     method="oidc",
#     k8s_api_host="https://api.example.com:6443",
#     oidc_issuer="https://your-oidc-provider.com",
#     client_id="your-client-id",
#     use_device_flow=True,
# )

# Create the entrypoint — this authenticates and sets up the SDK
cf = Codeflare(config=SDKConfig(
    auth=auth_config,
    namespace="your-namespace",  # default namespace for all operations
    log_level="INFO",
))

print(f"Connected. Default namespace: {cf.config.namespace}")

## Step 2: List existing clusters

Use `cf.clusters.list()` to see what Ray clusters are already running.
The namespace defaults to the one you set in `SDKConfig`.

In [ ]:
cf.clusters.list()

## Step 3: Create and bring up a Ray cluster

Use `cf.clusters.create()` to build a `Cluster` object. All keyword arguments
are forwarded to `ClusterConfiguration`. The namespace is inherited from
`SDKConfig` unless you override it.

In [ ]:
cluster = cf.clusters.create(
    name="entrypoint-demo",
    num_workers=2,
    head_cpu_requests="500m",
    head_cpu_limits="500m",
    head_memory_requests=5,
    head_memory_limits=8,
    worker_cpu_requests="250m",
    worker_cpu_limits=1,
    worker_memory_requests=4,
    worker_memory_limits=6,
    head_extended_resource_requests={"nvidia.com/gpu": 0},
    worker_extended_resource_requests={"nvidia.com/gpu": 0},
    write_to_file=False,
    # local_queue="your-local-queue",  # optional — auto-detected if omitted
)

In [ ]:
cluster.apply()

In [ ]:
cluster.wait_ready()

In [ ]:
cluster.details()

## Step 4: Submit a RayJob CR via `cf.jobs`

Use `cf.jobs.submit()` to create and immediately submit a RayJob.
The job runs on a short-lived managed cluster (Kuberay handles lifecycle).

In [ ]:
from codeflare_sdk import ManagedClusterConfig

job = cf.jobs.submit(
    name="entrypoint-demo-job",
    entrypoint="python -c 'import ray; ray.init(); print(\"Hello from the Codeflare entrypoint!\")'",
    cluster_config=ManagedClusterConfig(
        head_memory_requests=6,
        head_memory_limits=8,
        num_workers=1,
        worker_cpu_requests=1,
        worker_cpu_limits=1,
        worker_memory_requests=4,
        worker_memory_limits=6,
    ),
)

In [ ]:
# Check job status (may show 'unknown' while the managed cluster spins up)
job.status()

## Step 5: Retrieve an existing cluster

`cf.clusters.get()` fetches a running cluster by name — useful when
reconnecting to a cluster you created earlier.

In [ ]:
same_cluster = cf.clusters.get(name="entrypoint-demo")
same_cluster.status()

## Step 6: Clean up

Bring down the cluster. The managed RayJob cluster is cleaned up
automatically by Kuberay when the job completes.

In [ ]:
cluster.down()

In [ ]:
# No explicit logout needed — authentication is managed by kube-authkit